# 🌱 Proyecto Final — Análisis de la Transición Energética en Colombia
**Programa:** Talento Tech | Análisis de Datos — Nivel Integrador  
**Universidad:** Universidad de Caldas  
**Tema:** Optimización de la matriz eléctrica hacia fuentes renovables  

---
## Objetivo
Analizar el estado actual y la evolución de la transición energética en Colombia,
identificando patrones de inversión, generación, cobertura y reducción de emisiones
de CO₂ en los principales departamentos del país.

In [ ]:
# ── Librerías ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import warnings
warnings.filterwarnings('ignore')

# Estilo visual
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11
})

VERDE  = '#0D9488'
AZUL   = '#065A82'
AMBAR  = '#D97706'
GRIS   = '#64748B'
ROJO   = '#DC2626'
VERDE2 = '#16A34A'

print('✅ Librerías cargadas correctamente')

## 1. Carga y Preparación de Datos
Se utilizan datos sintéticos realistas basados en fuentes oficiales:
UPME, XM (operador del mercado), IRENA y DANE.

In [ ]:
# ── DATOS DE PLANTAS DE GENERACIÓN ────────────────────────
df_plantas = pd.DataFrame({
    'planta': ['Hidroituango','Porce III','Chivor','El Quimbo','Betania',
               'Sogamoso','Miel I','Termocol','Termosierra','Solar La Loma',
               'Guajira I (Eólica)','Jepírachi','Solar Acacías','Alpha Solar',
               'Biomasa Sucromiles','PCH Morro Azul','PCH Consota',
               'H2 Verde Piloto','Eólico Alta Guajira','Solar Sabana Centro'],
    'fuente': ['Hidroeléctrica','Hidroeléctrica','Hidroeléctrica','Hidroeléctrica',
               'Hidroeléctrica','Hidroeléctrica','Hidroeléctrica','Carbón','Gas Natural',
               'Solar FV','Eólica','Eólica','Solar FV','Solar FV','Biomasa',
               'PCH','PCH','Hidrógeno Verde','Eólica','Solar FV'],
    'tipo': ['Renovable','Renovable','Renovable','Renovable','Renovable',
              'Renovable','Renovable','No Renovable','No Renovable',
              'Renovable','Renovable','Renovable','Renovable','Renovable',
              'Renovable','Renovable','Renovable','Transición','Renovable','Renovable'],
    'capacidad_mw': [2400,660,1000,400,540,820,396,330,460,202,300,19.5,100,50,25,19.8,18.5,0.5,200,150],
    'departamento': ['Antioquia','Antioquia','Cundinamarca','Huila','Huila',
                     'Santander','Caldas','Valle del Cauca','Cesar','Cesar',
                     'La Guajira','La Guajira','Meta','Bolívar','Valle del Cauca',
                     'Risaralda','Risaralda','Atlántico','La Guajira','Cundinamarca'],
    'año_entrada': [2022,2010,1977,2015,1987,2014,2002,1998,2006,2021,
                    2022,2004,2023,2023,2018,2015,2012,2024,2025,2025],
    'estado': ['Operativa']*17 + ['En construcción','En construcción','Proyectada']
})

# ── INDICADORES DEPARTAMENTALES 2020-2024 ─────────────────
deptos = ['Antioquia','Caldas','La Guajira','Atlántico','Valle del Cauca']
años   = [2020,2021,2022,2023,2024]

consumo = {
    'Antioquia':     [14200,14600,15100,15500,16000],
    'Caldas':        [2100, 2150, 2220, 2290, 2360],
    'La Guajira':    [980,  1020, 1080, 1150, 1230],
    'Atlántico':     [4200, 4350, 4510, 4680, 4850],
    'Valle del Cauca':[9800,10050,10350,10650,10950]
}
cobertura = {
    'Antioquia':      [97.5,97.8,98.1,98.4,98.7],
    'Caldas':         [96.2,96.5,96.9,97.2,97.5],
    'La Guajira':     [78.3,79.5,81.2,83.8,86.5],
    'Atlántico':      [95.1,95.4,95.8,96.2,96.6],
    'Valle del Cauca':[97.8,98.0,98.3,98.5,98.7]
}
emisiones = {
    'Antioquia':      [82.4,80.1,75.3,70.2,65.8],
    'Caldas':         [45.2,43.8,41.0,38.5,36.1],
    'La Guajira':     [210.5,195.0,170.2,140.5,108.3],
    'Atlántico':      [180.3,172.5,160.0,142.5,125.8],
    'Valle del Cauca':[95.5,91.2,85.8,80.3,75.1]
}
inversion = {
    'Antioquia':      [120.5,145.2,180.0,220.0,280.0],
    'Caldas':         [18.5, 22.0, 28.5, 35.0, 42.0],
    'La Guajira':     [45.0, 68.0,120.0,185.0,250.0],
    'Atlántico':      [30.0, 38.0, 55.0, 80.0,110.0],
    'Valle del Cauca':[80.0, 95.0,120.0,150.0,190.0]
}

rows = []
for d in deptos:
    for i, y in enumerate(años):
        rows.append({'departamento': d, 'año': y,
                     'consumo_gwh': consumo[d][i],
                     'cobertura_pct': cobertura[d][i],
                     'emision_tco2_gwh': emisiones[d][i],
                     'inversion_mmusd': inversion[d][i]})

df_ind = pd.DataFrame(rows)

print('Plantas cargadas:', len(df_plantas))
print('Registros de indicadores:', len(df_ind))
df_plantas.head()

## 2. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Estadísticas básicas
print('=== CAPACIDAD INSTALADA POR TIPO ===')
cap_tipo = df_plantas[df_plantas.estado=='Operativa'].groupby('tipo')['capacidad_mw'].agg(['sum','count','mean'])
cap_tipo.columns = ['MW Total','Plantas','MW Promedio']
cap_tipo['% del Total'] = (cap_tipo['MW Total'] / cap_tipo['MW Total'].sum() * 100).round(1)
print(cap_tipo.round(1))

print('\n=== CAPACIDAD POR FUENTE ===')
cap_fuente = df_plantas[df_plantas.estado=='Operativa'].groupby('fuente')['capacidad_mw'].sum().sort_values(ascending=False)
print(cap_fuente.round(1))

In [ ]:
# GRÁFICA 1: Composición de la Matriz Eléctrica
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Composición de la Matriz Eléctrica Colombiana (Plantas Operativas)', 
             fontsize=14, fontweight='bold', y=1.02)

# Pie chart - por tipo
op = df_plantas[df_plantas.estado=='Operativa']
tipo_sum = op.groupby('tipo')['capacidad_mw'].sum()
colores_tipo = [VERDE, AMBAR, AZUL]
wedges, texts, autotexts = axes[0].pie(
    tipo_sum.values,
    labels=tipo_sum.index,
    autopct='%1.1f%%',
    colors=colores_tipo,
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'linewidth': 2, 'edgecolor': 'white'}
)
for t in autotexts:
    t.set_fontsize(11)
    t.set_fontweight('bold')
    t.set_color('white')
axes[0].set_title('Por tipo de fuente (MW instalados)', pad=10)

# Barras horizontales - por fuente
fuente_sum = op.groupby('fuente')['capacidad_mw'].sum().sort_values()
colores_bar = [VERDE if f not in ['Carbón','Gas Natural','Petróleo/Diesel'] else ROJO 
               for f in fuente_sum.index]
bars = axes[1].barh(fuente_sum.index, fuente_sum.values, color=colores_bar, edgecolor='white', linewidth=0.5)
for bar, val in zip(bars, fuente_sum.values):
    axes[1].text(val + 30, bar.get_y() + bar.get_height()/2,
                 f'{val:,.0f} MW', va='center', fontsize=9, color=GRIS)
axes[1].set_title('Capacidad instalada por fuente (MW)', pad=10)
axes[1].set_xlabel('Megavatios (MW)')
patch_ren = mpatches.Patch(color=VERDE, label='Renovable')
patch_nor = mpatches.Patch(color=ROJO, label='No Renovable')
axes[1].legend(handles=[patch_ren, patch_nor], loc='lower right')

plt.tight_layout()
plt.savefig('/home/claude/grafica_matriz.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: grafica_matriz.png')

In [ ]:
# GRÁFICA 2: Evolución de emisiones e inversión por departamento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Evolución 2020–2024: Emisiones e Inversión por Departamento',
             fontsize=13, fontweight='bold')

colores_dep = [VERDE, AZUL, AMBAR, ROJO, GRIS]

for i, dep in enumerate(deptos):
    datos = df_ind[df_ind.departamento == dep]
    axes[0].plot(datos.año, datos.emision_tco2_gwh, marker='o', linewidth=2,
                 color=colores_dep[i], label=dep)
    axes[1].plot(datos.año, datos.inversion_mmusd, marker='s', linewidth=2,
                 color=colores_dep[i], label=dep)

axes[0].set_title('Emisiones de CO₂ (tonCO₂/GWh)')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('tCO₂ por GWh')
axes[0].legend(fontsize=8)
axes[0].set_xticks(años)

axes[1].set_title('Inversión en Energías Limpias (MM USD)')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('Millones USD')
axes[1].legend(fontsize=8)
axes[1].set_xticks(años)

plt.tight_layout()
plt.savefig('/home/claude/grafica_emision_inversion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: grafica_emision_inversion.png')

In [ ]:
# GRÁFICA 3: Cobertura eléctrica por departamento
fig, ax = plt.subplots(figsize=(10, 5))

for i, dep in enumerate(deptos):
    datos = df_ind[df_ind.departamento == dep]
    ax.fill_between(datos.año, datos.cobertura_pct, alpha=0.15, color=colores_dep[i])
    ax.plot(datos.año, datos.cobertura_pct, marker='o', linewidth=2.5,
            color=colores_dep[i], label=dep)
    ax.annotate(f"{datos[datos.año==2024].cobertura_pct.values[0]}%",
                xy=(2024, datos[datos.año==2024].cobertura_pct.values[0]),
                xytext=(2024.15, datos[datos.año==2024].cobertura_pct.values[0]),
                fontsize=8.5, color=colores_dep[i], fontweight='bold')

ax.axhline(y=100, color=GRIS, linestyle='--', linewidth=1, alpha=0.5, label='Meta 100%')
ax.set_title('Cobertura Eléctrica por Departamento (% hogares)', fontsize=13, fontweight='bold')
ax.set_xlabel('Año')
ax.set_ylabel('% de hogares con acceso eléctrico')
ax.set_xticks(años)
ax.set_ylim(70, 101)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('/home/claude/grafica_cobertura.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: grafica_cobertura.png')

In [ ]:
# ── ANÁLISIS ESTADÍSTICO AVANZADO ─────────────────────────
print('=== REDUCCIÓN DE EMISIONES 2020→2024 ===')
reduccion = []
for dep in deptos:
    e20 = df_ind[(df_ind.departamento==dep) & (df_ind.año==2020)].emision_tco2_gwh.values[0]
    e24 = df_ind[(df_ind.departamento==dep) & (df_ind.año==2024)].emision_tco2_gwh.values[0]
    pct = (e20 - e24) / e20 * 100
    reduccion.append({'Departamento': dep, 'Emisión 2020': e20, 'Emisión 2024': e24, 'Reducción %': round(pct, 1)})

df_red = pd.DataFrame(reduccion).sort_values('Reducción %', ascending=False)
print(df_red.to_string(index=False))

print('\n=== TASA CRECIMIENTO INVERSIÓN (CAGR) ===')
for dep in deptos:
    i20 = df_ind[(df_ind.departamento==dep) & (df_ind.año==2020)].inversion_mmusd.values[0]
    i24 = df_ind[(df_ind.departamento==dep) & (df_ind.año==2024)].inversion_mmusd.values[0]
    cagr = ((i24 / i20) ** (1/4) - 1) * 100
    print(f'  {dep:22s}  CAGR: {cagr:.1f}% anual  ({i20:.0f} → {i24:.0f} MM USD)')

In [ ]:
# GRÁFICA 4: Dispersión — Inversión vs Reducción de Emisiones
inv_2024 = {dep: df_ind[(df_ind.departamento==dep) & (df_ind.año==2024)].inversion_mmusd.values[0] for dep in deptos}
red_pct  = {r['Departamento']: r['Reducción %'] for _, r in df_red.iterrows()}

fig, ax = plt.subplots(figsize=(9, 6))

for i, dep in enumerate(deptos):
    cob = df_ind[(df_ind.departamento==dep) & (df_ind.año==2024)].cobertura_pct.values[0]
    sc = ax.scatter(inv_2024[dep], red_pct[dep],
               s=cob * 18,
               color=colores_dep[i], alpha=0.8, edgecolors='white', linewidth=1.5,
               label=dep, zorder=5)
    ax.annotate(dep, (inv_2024[dep], red_pct[dep]),
                textcoords='offset points', xytext=(8, 4),
                fontsize=9.5, color=colores_dep[i], fontweight='bold')

ax.set_xlabel('Inversión en Energías Limpias 2024 (MM USD)', fontsize=11)
ax.set_ylabel('Reducción de Emisiones CO₂ 2020→2024 (%)', fontsize=11)
ax.set_title('Inversión vs Reducción de Emisiones\n(tamaño = cobertura eléctrica 2024)',
             fontsize=12, fontweight='bold')

# Línea de tendencia
xs = np.array(list(inv_2024.values()))
ys = np.array([red_pct[d] for d in deptos])
z  = np.polyfit(xs, ys, 1)
p  = np.poly1d(z)
ax.plot(np.sort(xs), p(np.sort(xs)), '--', color=GRIS, alpha=0.6, linewidth=1.5, label='Tendencia')
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('/home/claude/grafica_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: grafica_scatter.png')

In [ ]:
# GRÁFICA 5: Proyección capacidad renovable 2024-2030
años_proy = np.array([2024, 2025, 2026, 2027, 2028, 2029, 2030])

# Capacidad actual operativa por tipo
cap_actual_renov = df_plantas[df_plantas.tipo=='Renovable']['capacidad_mw'].sum()
# Proyección: crecimiento basado en pipeline UPME
cap_hidro  = np.array([7416, 7416, 7416, 7816, 7816, 8016, 8016])  # poco crecimiento
cap_solar  = np.array([502,  702,  950, 1250, 1600, 2000, 2500])
cap_eolico = np.array([519,  719,  919, 1219, 1619, 2119, 2719])
cap_otros  = np.array([478,  490,  510,  530,  550,  570,  600])
cap_total  = cap_hidro + cap_solar + cap_eolico + cap_otros

fig, ax = plt.subplots(figsize=(12, 5))

ax.stackplot(años_proy,
             [cap_hidro, cap_solar, cap_eolico, cap_otros],
             labels=['Hidroeléctrica','Solar FV','Eólica','Otros Renovables'],
             colors=[AZUL, AMBAR, VERDE, GRIS],
             alpha=0.82)

for y, v in zip(años_proy, cap_total):
    ax.text(y, v + 150, f'{v:,.0f}\nMW', ha='center', fontsize=8.5,
            fontweight='bold', color='#1E293B')

ax.axhline(y=11200, color=ROJO, linestyle='--', linewidth=2, label='Meta Ley 2099 (11.200 MW FNCER)')
ax.set_title('Proyección Capacidad Renovable Instalada Colombia 2024–2030', 
             fontsize=13, fontweight='bold')
ax.set_ylabel('Megavatios (MW)')
ax.set_xlabel('Año')
ax.legend(loc='upper left', fontsize=9)
ax.set_xticks(años_proy)

plt.tight_layout()
plt.savefig('/home/claude/grafica_proyeccion.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfica guardada: grafica_proyeccion.png')

## 3. Conclusiones y Recomendaciones

### Hallazgos Principales
1. **La Guajira** muestra la mayor reducción de emisiones (–48.6%) gracias a la expansión eólica y solar, confirmando su potencial como hub de energía limpia.
2. La **inversión en energías limpias** creció en todos los departamentos analizados, con CAGR promedio superior al 20% anual entre 2020 y 2024.
3. La **cobertura eléctrica** presenta una brecha crítica: La Guajira (86.5%) vs Antioquia (98.7%), requiriendo modelos de generación distribuida.
4. La **hidroeléctrica** sigue siendo el pilar (>80% de capacidad operativa), pero la solar y eólica muestran el mayor crecimiento relativo.
5. Colombia está **en camino** de alcanzar la meta de 11.2 GW de FNCER para 2030, si mantiene el ritmo actual de subastas.

### Recomendaciones
- Priorizar proyectos solares distribuidos en municipios con cobertura < 90%.
- Acelerar la cadena de valor del hidrógeno verde en zonas costeras (Atlántico, Bolívar).
- Implementar mecanismos de comunidades energéticas bajo el Decreto 829/2023.
- Fortalecer la interconexión eléctrica entre regiones para optimizar el despacho renovable.